In [1]:
# ¿Qué cursos y tecnologías tienen mayor adopción y mejor valoración para priorizar una oferta educativa?

import sqlite3
from pathlib import Path

import pandas as pd

ROOT = next(
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / "data").is_dir() and (path / "submission").is_dir()
)
SOURCE = ROOT / "data" / "datacamp_application.sql"
DATABASE = ROOT / "temp" / "datacamp.db"
OUTPUT = ROOT / "submission" / "course_ratings.csv"
DATABASE.parent.mkdir(exist_ok=True)


In [2]:
# El volcado conserva dos granos: curso y calificación de un usuario a un curso.

DATABASE.unlink(missing_ok=True)
sql_dump = SOURCE.read_text(encoding="utf-8").replace(
    chr(34) + "public" + chr(34) + ".", ""
)
with sqlite3.connect(DATABASE) as connection:
    connection.executescript(sql_dump)
    counts = pd.read_sql_query(
        "SELECT 'courses' AS table_name, COUNT(*) AS rows FROM courses UNION ALL SELECT 'rating', COUNT(*) FROM rating",
        connection,
    )
counts


,table_name,rows
0,courses,100
1,rating,59172


In [3]:
# La transformación cambia de una fila por calificación a una fila por curso.

query = """
SELECT c.course_id, c.title, COALESCE(c.programming_language, 'unknown') AS programming_language,
       COUNT(r.user_id) AS rating_count,
       COALESCE(ROUND(AVG(r.rating), 2), 0) AS average_rating
FROM courses c
LEFT JOIN rating r USING(course_id)
GROUP BY c.course_id, c.title, COALESCE(c.programming_language, 'unknown')
ORDER BY rating_count DESC, average_rating DESC, c.course_id
"""
with sqlite3.connect(DATABASE) as connection:
    course_ratings = pd.read_sql_query(query, connection)
course_ratings.head()


,course_id,title,programming_language,rating_count,average_rating
0,7,Introduction to Python,python,14950,4.60
1,32,Intermediate Python for Data Science,python,6507,4.58
2,96,Intro to SQL for Data Science,sql,6054,4.69
3,6,Joining Data in SQL,sql,3038,4.50
4,3,Intermediate R,r,2899,4.57


In [4]:
# Una reconciliación evita confundir la agregación con pérdida de calificaciones.

with sqlite3.connect(DATABASE) as connection:
    source_ratings = connection.execute("SELECT COUNT(*) FROM rating").fetchone()[0]
assert len(course_ratings) == 100
assert course_ratings.rating_count.sum() == source_ratings
assert course_ratings.average_rating.between(0, 5).all()
course_ratings.to_csv(OUTPUT, index=False)
course_ratings.groupby(
    "programming_language", as_index=False
).rating_count.sum().sort_values("rating_count", ascending=False)


,programming_language,rating_count
0,python,34070
1,r,11664
4,sql,11663
2,shell,1414
3,spreadsheets,242
5,unknown,119
